In [0]:
%run ./04_utils

In [0]:
# =========================================================
# NOTEBOOK : 02 - SILVER LAYER
# PURPOSE  : Cleaning + Type Casting + KPI Calculations
# =========================================================

# =========================================================
# 1. IMPORTS
# =========================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, col
import logging

# ✅ 04_utils se import karo — khud likhne ki zaroorat nahi
# from utils_04 import upsert_delta

# =========================================================
# 2. LOGGING
# =========================================================

logger = logging.getLogger("ecommerce_pipeline.silver")

# =========================================================
# 3. PATHS
# =========================================================

BRONZE_PATH = "/Volumes/workspace/default/sales_volume/bronze/data"
SILVER_PATH = "/Volumes/workspace/default/sales_volume/silver/"

# =========================================================
# 4. SILVER FUNCTION
# =========================================================

def run_silver(spark):

    try:

        logger.info("Starting Silver Layer")
        

        # --------------------------------------------------
        # Bronze se Data Read karo
        # --------------------------------------------------

        silver_df = spark.read.format("delta").load(BRONZE_PATH)

        # --------------------------------------------------
        # Type Casting (CSV me sab STRING hota hai)
        # --------------------------------------------------

        silver_df = silver_df \
            .withColumn("quantity", col("quantity").cast("integer")) \
            .withColumn("price",    col("price").cast("double")) \
            .withColumn("discount", col("discount").cast("double"))

        # --------------------------------------------------
        # Data Cleaning + Deduplication
        # --------------------------------------------------

        silver_df = silver_df \
            .filter(col("quantity") > 0) \
            .dropDuplicates(["order_id"])

        # --------------------------------------------------
        # KPI Calculations
        # --------------------------------------------------

        silver_df = silver_df \
            .withColumn("revenue",
                col("quantity") * col("price")
            ) \
            .withColumn("discount_amount",
                (col("quantity") * col("price") * col("discount")) / 100
            ) \
            .withColumn("final_revenue",
                col("revenue") - col("discount_amount")
            ) \
            .withColumn("processed_time", current_timestamp())

        # --------------------------------------------------
        # Data Quality Check
        # --------------------------------------------------

        invalid_count = silver_df.filter(
            col("price").isNull() | col("quantity").isNull()
        ).count()

        logger.info(f"Invalid Records Count : {invalid_count}")
        print(f"Invalid Records Count : {invalid_count}")

        # --------------------------------------------------
        # ✅ utils se import kiya — yahan seedha use karo
        # --------------------------------------------------

        upsert_delta(
            spark,
            silver_df,
            SILVER_PATH,
            "target.order_id = source.order_id"
        )
        
        logger.info("Silver Layer Completed Successfully")

    except Exception as e:

        logger.error(f"Silver Layer Failed : {str(e)}")
        raise


# =========================================================
# 5. DIRECT RUN
# =========================================================

if __name__ == "__main__":

    spark = SparkSession.builder \
        .appName("Silver-Layer") \
        .config("spark.sql.shuffle.partitions", "8") \
        .config("spark.databricks.delta.optimizeWrite.enabled", "true") \
        .config("spark.databricks.delta.autoCompact.enabled", "true") \
        .getOrCreate()

    run_silver(spark)